# 11 — LLM + MCP generico su Resource Retrieval

Questo notebook collega il nuovo layer **GTE → Qdrant → RetrievalService → MCP** al livello LLM.

L'obiettivo non è rendere *ogni dettaglio* completamente generico: il **core di orchestrazione è domain-agnostic**, mentre vocabolario, filtri e istruzioni restano in un `ResourceDomainProfile`. Questo rende il passaggio a WhoTeach un adattamento controllato del dominio, non una riscrittura dell'architettura.

Pipeline testata:

```text
User request
   ↓
Planner LLM
   ↓  semantic_query + domain-native filters
MCPResourceToolGateway
   ↓
search_resources
   ↓
RetrievalService → GTE → Qdrant
   ↓
Grounded resources
   ↓
Answer LLM
   ↓
Grounded answer + recommendations
```

Il profilo tecnico di retrieval resta **server-side**: il Planner non sceglie Qdrant, GTE, reranker o candidate pool.

## 0. Prerequisiti

Prima di eseguire questo notebook devono essere completati gli step del notebook 10:

- collection Qdrant v2 popolata con `document__D_full_enriched_v2`;
- embedding GTE multilingual base;
- package `movie_rag` installabile dalla root;
- MCP installato;
- configurazione LLM precedente ancora disponibile (`configs/llm_profiles_v1.json`);
- `GEMINI_API_KEY` nel `.env`.

Per il test interattivo usiamo il profilo `movielens_gte_fast_v2`, quindi **nessun reranker locale**.

In [1]:

%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise FileNotFoundError("Root del progetto non trovata: manca src/movie_rag")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=False)

print("Project root:", PROJECT_ROOT)
print("Gemini key available:", bool(os.getenv("GEMINI_API_KEY")))


Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
Gemini key available: True


## 1. Configurazione

Separiamo tre livelli:

1. **LLM profile** — provider/modello/temperature già definiti nel vecchio `llm_profiles_v1.json`;
2. **domain profile** — semantica del dominio e filtri ammessi;
3. **MCP retrieval profile** — collection Qdrant, encoder e profilo `fast/quality`, invisibili alla LLM.

In [ ]:
LLM_CONFIG_PATH = PROJECT_ROOT / "configs" / "llm_profiles_v1.json"
DOMAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "resource_domains_v1.json"
MCP_CONFIG_PATH = PROJECT_ROOT / "configs" / "resource_mcp_profiles_v1.json"

DOMAIN_ID = "movielens_v2"
RESOURCE_RAG_PROFILE = "movielens_gte_fast_v2"
# RESOURCE_RAG_PROFILE_RERANK = "movielens_gte_quality_v2"
WORKFLOW_ID = "grounded_resource_v1"

PLANNER_LLM_PROFILE = "gemini_free"
ANSWER_LLM_PROFILE = "gemini_free"
QUERY_REWRITE_STRATEGY = "translate_to_catalog_language"

USER_QUERY = (
    "Vorrei un film malinconico sulla memoria e l'identità, "
    "non horror e sotto le due ore."
)

os.environ["RESOURCE_RAG_PROFILE"] = RESOURCE_RAG_PROFILE
os.environ.setdefault("QDRANT_URL", "http://localhost:6333")


'http://localhost:6333'

## 2. Caricamento dei profili di dominio e retrieval

Il dominio MovieLens resta specifico dove deve esserlo: nomi dei filtri, descrizione del catalogo e istruzioni. Il core del workflow non contiene alcun riferimento a film.

In [4]:
from movie_rag.resource_domain import load_resource_domain_catalog
from movie_rag.resource_mcp_profiles import (
    ResourceSearchServiceRegistry,
    load_resource_mcp_profiles,
)


domain_catalog = load_resource_domain_catalog(DOMAIN_CONFIG_PATH)
domain = domain_catalog.get(DOMAIN_ID)

mcp_profile_catalog = load_resource_mcp_profiles(MCP_CONFIG_PATH)
mcp_profile = mcp_profile_catalog.get(RESOURCE_RAG_PROFILE)

assert mcp_profile.domain_id == domain.domain_id
assert not domain.template_only

print("Domain:", domain.domain_id)
print("Allowed filters:", domain.allowed_filter_keys)
print("MCP profile:", mcp_profile.profile_id)
print("Collection:", mcp_profile.collection_name)
print("Search profile:", mcp_profile.search_profile_id)
print("Encoder:", mcp_profile.encoder.model_name)


Domain: movielens_v2
Allowed filters: ['max_runtime', 'min_runtime', 'min_year', 'max_year', 'languages', 'countries', 'include_genres', 'exclude_genres', 'adult']
MCP profile: movielens_gte_quality_v2
Collection: movies_d_full_enriched_v2_gte_multilingual_base_v1
Search profile: quality
Encoder: Alibaba-NLP/gte-multilingual-base


## 3. Costruzione del RetrievalService da configurazione

Il registry costruisce encoder GTE, `QdrantResourceRetriever` e `RetrievalService`. La LLM non vede questi dettagli.

In [5]:
registry = ResourceSearchServiceRegistry.from_environment(
    MCP_CONFIG_PATH,
    profile_id=RESOURCE_RAG_PROFILE,
)

print("Retriever:", registry.service.retriever.name)
print("Available search profiles:", list(registry.service.profiles))
print("Server default profile:", registry.profile.search_profile_id)

print("MCP profile:", registry.profile.profile_id)
print("Search profile:", registry.profile.search_profile_id)
print("Reranker:", registry.service.reranker)


Retriever: gte_multilingual_base_dense
Available search profiles: ['fast', 'quality']
Server default profile: quality
MCP profile: movielens_gte_quality_v2
Search profile: quality
Reranker: <movie_rag.resource_service.ScoreRerankerAdapter object at 0x14612b770>


## 4. MCP generico

`search_resources` espone soltanto `query`, `filters` e `limit`.

Il parametro `profile` **non è visibile alla LLM** in questa fase: `fast`/`quality` è una scelta del server. In futuro, se l'esperimento sull'orchestrazione dinamica lo richiederà, potremo abilitarlo esplicitamente.

In [6]:
from movie_rag.resource_adapters import movie_filters_to_resource_filters
from movie_rag.resource_mcp import create_resource_mcp_server


mcp_server = create_resource_mcp_server(
    registry.service,
    name="MovieLens v2 Resource Retrieval",
    filter_adapter=movie_filters_to_resource_filters,
    default_profile_id=registry.profile.search_profile_id,
    expose_profile_to_client=False,
    exact=registry.profile.exact,
)

print("MCP server ready")


MCP server ready


## 5. Provider LLM

Riutilizziamo l'adapter Gemini già presente nel progetto. Il nuovo core LLM dipende soltanto dal contratto `generate_text(...)`, quindi in seguito possiamo sostituire il provider senza cambiare workflow, MCP o retrieval.

In [8]:
from movie_rag import GeminiProvider, load_llm_config


llm_catalog = load_llm_config(LLM_CONFIG_PATH)
planner_provider = GeminiProvider(llm_catalog.get_llm_profile(PLANNER_LLM_PROFILE))
answer_provider = GeminiProvider(llm_catalog.get_llm_profile(ANSWER_LLM_PROFILE))

print("Planner profile:", PLANNER_LLM_PROFILE)
print("Answer profile:", ANSWER_LLM_PROFILE)


Planner profile: gemini_free
Answer profile: gemini_free


## 6. Test isolato del Planner

Il Planner deve produrre soltanto:

- query semantica;
- filtri **nativi del dominio**;
- eventuale richiesta di chiarimento.

Non può scegliere modello di embedding, Qdrant, reranker o profilo tecnico.

In [9]:
from movie_rag.resource_llm_agents import ResourcePlannerAgent


planner = ResourcePlannerAgent(
    provider=planner_provider,
    domain=domain,
    temperature=0.0,
    query_rewrite_strategy=QUERY_REWRITE_STRATEGY,
)

plan, planner_trace = await planner.run(USER_QUERY)
display(plan.model_dump())
display(pd.DataFrame([planner_trace.model_dump(mode="json")]))


[10/08/26 17:06:38] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=4028100;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=4028101;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'semantic_query': 'melancholic movie about memory and identity, personal reflection, psychological drama',
 'filters': {'max_runtime': 120, 'exclude_genres': ['Horror']},
 'clarification_required': False,
 'clarification_question': None,
 'rationale': 'The user is looking for a melancholic film about memory and identity, excluding horror, with a runtime under two hours (120 minutes). These requirements map directly to semantic concepts and explicit runtime/genre filters.'}

,role,prompt_version,provider_trace
0,planner,resource_planner_v1,"{'agent': 'resource_planner', 'provider': 'gem..."


### Controllo atteso

Per la query di esempio ci aspettiamo indicativamente:

```text
semantic_query: contiene malinconia, memoria, identità
filters.max_runtime: 120
filters.exclude_genres: ["Horror"]
```

Il testo preciso della query semantica può variare con il modello.

## 7. End-to-end: Planner → MCP/Qdrant → Answer

Ora eseguiamo il workflow completo. Il gateway valida l'output del tool MCP con i modelli Pydantic generici.

In [10]:
from mcp import Client

from movie_rag.resource_llm_workflow import build_resource_assistant_workflow
from movie_rag.resource_mcp_gateway import MCPResourceToolGateway


async def run_resource_query(query: str):
    async with Client(mcp_server, raise_exceptions=True) as client:
        gateway = MCPResourceToolGateway(client)
        workflow = build_resource_assistant_workflow(
            domain=domain,
            planner_provider=planner_provider,
            answer_provider=answer_provider,
            gateway=gateway,
            workflow_id=WORKFLOW_ID,
            result_limit=10,
            query_rewrite_strategy=QUERY_REWRITE_STRATEGY,
        )
        return await workflow.run(query)


state = await run_resource_query(USER_QUERY)
print({"turn_id": state.turn_id, "status": state.status})


[10/08/26 17:06:49] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=4028106;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=4028107;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

[10/08/26 17:07:18] INFO     No device provided, using mps                                             ]8;id=4028114;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=4028115;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#190\190]8;;\

[10/08/26 17:07:19] INFO     Loading SentenceTransformer model from                                   ]8;id=4028121;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=4028122;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#1022\1022]8;;\
                             Alibaba-NLP/gte-multilingual-base.                                                    

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[10/08/26 17:07:27] INFO     HTTP Request: POST                                                     ]8;id=4028129;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4028130;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_v2_gte_multil                
                             ingual_base_v1/points/query "HTTP/1.1 200 OK"                                         

[10/08/26 17:07:28] INFO     No modules.json found for Alibaba-NLP/gte-multilingual-reranker-base,    ]8;id=4028136;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=4028137;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#1011\1011]8;;\
                             initializing a new CrossEncoder model.                                                

[10/08/26 17:07:36] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=4028142;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=4028143;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'turn_id': '0d04206abefc4ba6a9ae146ab0bbd098', 'status': 'completed'}


## 8. Ispezione del risultato

La risposta è genericamente espressa in termini di `resource_id`, `title`, `text`, `metadata`. I campi cinematografici restano dentro `metadata` e non sono parte del contratto core.

In [11]:
display(Markdown("### Piano"))
display(state.plan.model_dump() if state.plan else {})

if state.search_response is not None:
    search = state.search_response
    print({
        "search_status": search.status,
        "query_to_mcp": search.query,
        "server_profile": search.pipeline.profile_id,
        "retriever": search.pipeline.retriever,
        "reranker": search.pipeline.reranker,
        "returned_count": search.returned_count,
    })
    rows = []
    for hit in search.results:
        rows.append({
            "rank": hit.rank,
            "resource_id": hit.resource_id,
            "title": hit.title,
            "score": hit.score,
            "year": hit.metadata.get("year"),
            "runtime": hit.metadata.get("runtime"),
            "genres": " | ".join(hit.metadata.get("genres", [])),
        })
    display(pd.DataFrame(rows))

if state.final_answer is not None:
    display(Markdown("### Risposta grounded"))
    display(Markdown(state.final_answer.answer))
    display(pd.DataFrame([r.model_dump() for r in state.final_answer.recommendations]))

print("Grounding violations:", state.grounding_violations)


### Piano

{'semantic_query': 'A melancholic film exploring themes of memory, personal identity, and introspection, avoiding horror elements.',
 'filters': {'max_runtime': 120, 'exclude_genres': ['Horror']},
 'clarification_required': False,
 'clarification_question': None,
 'rationale': "The user requested a melancholic movie about memory and identity, specifying it should not be horror and must be under two hours. The runtime constraint translates to max_runtime: 120, and the genre exclusion maps to exclude_genres: ['Horror']. The thematic elements are captured in the semantic_query."}

{'search_status': 'ok', 'query_to_mcp': 'A melancholic film exploring themes of memory, personal identity, and introspection, avoiding horror elements.', 'server_profile': 'quality', 'retriever': 'gte_multilingual_base_dense', 'reranker': 'gte_multilingual_reranker', 'returned_count': 2}


,rank,resource_id,title,score,year,runtime,genres
0,1,7361,Eternal Sunshine of the Spotless Mind,0.613917,2004.0,108.0,Science Fiction | Drama | Romance | Sci-Fi
1,2,4226,Memento,0.534047,2000.0,113.0,Mystery | Thriller


### Risposta grounded

Ecco un film che esplora i temi della memoria e dell'identità con toni malinconici, senza essere un horror e con una durata inferiore alle due ore.

,resource_id,title,reason
0,7361,Eternal Sunshine of the Spotless Mind,Il film affronta direttamente i temi della mem...


Grounding violations: []


## 9. Trace LLM e tool

Le trace del provider vengono conservate in forma generica. Le chiamate MCP registrano tool, argomenti e latenza.

In [12]:
display(pd.DataFrame([trace.model_dump(mode="json") for trace in state.llm_traces]))
display(pd.DataFrame([trace.model_dump(mode="json") for trace in state.tool_traces]))


,role,prompt_version,provider_trace
0,planner,resource_planner_v1,"{'agent': 'resource_planner', 'provider': 'gem..."
1,answer,resource_answer_v1,"{'agent': 'resource_answer', 'provider': 'gemi..."


,tool,latency_ms,arguments,ok
0,search_resources,22637.739875,{'query': 'A melancholic film exploring themes...,True


## 10. Test no-results e clarification

Due rami sono gestiti in modo deterministico:

- se il Planner dichiara che manca un'informazione indispensabile, MCP non viene chiamato;
- se Qdrant restituisce zero risultati, l'Answer LLM non inventa alternative fuori catalogo.

In [ ]:
RUN_EDGE_CASES = False

if RUN_EDGE_CASES:
    strict_state = await run_resource_query(
        "Vorrei un film italiano uscito prima del 1930, sotto i 40 minuti, non horror."
    )
    print(strict_state.status)
    display(strict_state.final_answer.model_dump())


[10/08/26 17:08:05] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=4028148;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=4028149;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

[10/08/26 17:08:07] INFO     HTTP Request: POST                                                     ]8;id=4028154;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4028155;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_v2_gte_multil                
                             ingual_base_v1/points/query "HTTP/1.1 200 OK"                                         

no_results


{'status': 'no_results',
 'answer': 'No grounded resources matched the current request and constraints. Try reformulating the request or explicitly relaxing a constraint.',
 'recommendations': []}

## 11. Profilo `quality`: previsto ma non necessario per lo sviluppo locale

Il file `resource_mcp_profiles_v1.json` contiene anche `movielens_gte_quality_v2`. Quel profilo usa `candidate_k=20`, `final_k=10`, `rerank=true`.

Per attivarlo bisogna costruire il registry passando un `ResourceReranker`. Sul Mac conviene continuare con `fast`; su un backend CUDA il medesimo `RetrievalService` può ricevere il reranker senza modificare Planner, MCP gateway o Answer agent.

## 12. Cosa cambia quando passeremo a WhoTeach

**Non vogliamo zero modifiche.** Sarebbe over-engineering e rischierebbe di nascondere differenze reali tra film e risorse educative.

Restano invariati:

- `ResourceDocument`;
- encoder/retriever Qdrant;
- `RetrievalService`;
- `search_resources`;
- `MCPResourceToolGateway`;
- workflow Planner → retrieval → Answer;
- provider LLM e trace.

Cambieranno deliberatamente:

1. adapter di ingestion/chunking WhoTeach;
2. collection e MCP profile;
3. `ResourceDomainProfile` (terminologia, filtri realmente disponibili, istruzioni);
4. eventualmente prompt/agent specializzati pedagogici aggiuntivi.

Il profilo `whoteach_template` è solo uno scheletro: **non va attivato finché non abbiamo congelato il vero payload dei chunk WhoTeach**.

## 13. Prossimo step dopo questo notebook

Una volta validato questo flusso, abbiamo una baseline fissa e domain-agnostic sufficiente:

```text
Planner → search_resources → Answer
```

Il prossimo lavoro di tesi può concentrarsi sull'orchestrazione vera: Elicitation, Planning didattico, Retrieval, Ranking/Validation e Explanation, confrontando una pipeline fissa con un orchestratore condizionale.